In [0]:
bronze_df = spark.read.format("delta").load(
    "dbfs:/EmployeeLifecycleProject/bronze/day_1"
)
bronze_df.show(5)

+------+-------+----------+------+----------+------+
|emp_id|   name|department|salary| join_date|status|
+------+-------+----------+------+----------+------+
|   151|Emp_151| Marketing| 73963|2020-03-08|Active|
|   152|Emp_152| Marketing| 79253|2020-03-31|Active|
|   153|Emp_153|        IT| 83668|2021-10-16|Active|
|   154|Emp_154|   Finance| 84331|2020-06-28|Active|
|   155|Emp_155| Marketing| 90570|2021-12-28|Active|
+------+-------+----------+------+----------+------+
only showing top 5 rows


In [0]:
bronze_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- join_date: date (nullable = true)
 |-- status: string (nullable = true)



In [0]:
print("Total Records:", bronze_df.count())

Total Records: 200


In [0]:
duplicate_c = bronze_df.count() - bronze_df.dropDuplicates().count()
print("Duplicate Records:", duplicate_c)

Duplicate Records: 0


In [0]:
if duplicate_c>0:
    silver_df = bronze_df.dropDuplicates()
    print(f"{duplicate_c} duplicate records removed.")
else:
    silver_df = bronze_df
    print("No duplicate records found.")

No duplicate records found.


In [0]:
from pyspark.sql.functions import col, sum, trim, when
silver_df.select([
    sum(
        when(
            col(c).isNull() | (trim(col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in silver_df.columns
]).show()

+------+----+----------+------+---------+------+
|emp_id|name|department|salary|join_date|status|
+------+----+----------+------+---------+------+
|     0|   0|         0|     0|        0|     0|
+------+----+----------+------+---------+------+



In [0]:
from pyspark.sql.functions import upper, col
silver_df = silver_df.withColumn("department", upper(col("department")))
silver_df = silver_df.withColumn("status", upper(col("status")))

In [0]:
silver_df.select("department").distinct().show()

+----------+
|department|
+----------+
| MARKETING|
|        HR|
|     SALES|
|        IT|
|   FINANCE|
+----------+



In [0]:
silver_df.select("status").distinct().show()

+------+
|status|
+------+
|ACTIVE|
+------+



In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("dbfs:/EmployeeLifecycleProject/silver/day_1")

In [0]:
check_df = spark.read.format("delta").load(
    "dbfs:/EmployeeLifecycleProject/silver/day_1"
)
check_df.show(5)

+------+------+----------+------+----------+------+
|emp_id|  name|department|salary| join_date|status|
+------+------+----------+------+----------+------+
|    51|Emp_51|        IT| 35250|2021-04-26|ACTIVE|
|    52|Emp_52| MARKETING| 34406|2021-03-11|ACTIVE|
|    53|Emp_53| MARKETING| 81684|2021-05-05|ACTIVE|
|    54|Emp_54|        IT| 89302|2021-05-22|ACTIVE|
|    55|Emp_55|        HR| 40890|2021-07-25|ACTIVE|
+------+------+----------+------+----------+------+
only showing top 5 rows


In [0]:
print("Total Records:", silver_df.count())
silver_df.describe().show()

Total Records: 200
+-------+------------------+------+----------+-----------------+------+
|summary|            emp_id|  name|department|           salary|status|
+-------+------------------+------+----------+-----------------+------+
|  count|               200|   200|       200|              200|   200|
|   mean|             100.5|  NULL|      NULL|         66215.84|  NULL|
| stddev|57.879184513951124|  NULL|      NULL|21259.44906404996|  NULL|
|    min|                 1| Emp_1|   FINANCE|            30320|ACTIVE|
|    max|               200|Emp_99|     SALES|            99945|ACTIVE|
+-------+------------------+------+----------+-----------------+------+



In [0]:
from pyspark.sql.functions import col, upper

days = ["day_2", "day_3", "day_4", "day_5"]

for day in days:
    print(f"Processing {day}")

    # Read Bronze
    bronze_df = spark.read.format("delta").load(
        f"dbfs:/EmployeeLifecycleProject/bronze/{day}"
    )

    # Remove duplicates
    silver_df = bronze_df.dropDuplicates()

    # Standardize text columns
    silver_df = silver_df.withColumn(
        "department",
        upper(col("department"))
    )

    silver_df = silver_df.withColumn(
        "status",
        upper(col("status"))
    )

    # Save Silver
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"dbfs:/EmployeeLifecycleProject/silver/{day}")

    print(f"{day} saved successfully.")

Processing day_2
day_2 saved successfully.
Processing day_3
day_3 saved successfully.
Processing day_4
day_4 saved successfully.
Processing day_5
day_5 saved successfully.


In [0]:
display(dbutils.fs.ls("dbfs:/EmployeeLifecycleProject/silver"))

path,name,size,modificationTime
dbfs:/EmployeeLifecycleProject/silver/day_1/,day_1/,0,1784430352000
dbfs:/EmployeeLifecycleProject/silver/day_2/,day_2/,0,1784430794000
dbfs:/EmployeeLifecycleProject/silver/day_3/,day_3/,0,1784430797000
dbfs:/EmployeeLifecycleProject/silver/day_4/,day_4/,0,1784430799000
dbfs:/EmployeeLifecycleProject/silver/day_5/,day_5/,0,1784430802000


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS employeelifecycleworkspace.default.employee_outputs;

In [0]:
silver_df = spark.read.format("delta").load(
    "dbfs:/EmployeeLifecycleProject/silver/day_1"
)
silver_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/employeelifecycleworkspace/default/employee_outputs/day_1")

In [0]:
days = ["day_2", "day_3", "day_4", "day_5"]

for day in days:
    df = spark.read.format("delta").load(
        f"dbfs:/EmployeeLifecycleProject/silver/{day}"
    )
    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"/Volumes/employeelifecycleworkspace/default/employee_outputs/{day}")

    print(f"{day} exported successfully.")

day_2 exported successfully.
day_3 exported successfully.
day_4 exported successfully.
day_5 exported successfully.
